***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.5 卷积：响应函数如何重塑信号](2_5_convolution.ipynb)
    * 下一节：[2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)

***


导入标准模块:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
HTML('../style/course.css') #apply general CSS

导入本节所需的专用模块:

In [ ]:
pass

6. [互相关、自相关与相似性度量](2_6_cross_correlation_and_auto_correlation.ipynb)
    1. [互相关](#math:sec:cross_correlation)
    2. [自相关](#math:sec:auto_correlation)


## 2.6 互相关、自相关与相似性度量<a id='math:sec:cross_correlation_and_auto_correlation'></a>

相关是干涉测量的操作核心。相关器并不直接输出图像，而是比较两个接收信号在不同延迟下的相似程度；互相关决定基线上的复可见度，自相关则与总功率、噪声表征以及谱估计密切相关。

因此，虽然本节篇幅不长，它却是后面理解相关器、功率谱、维纳-辛钦关系以及“为什么两个天线的联合测量会产生结构信息”的重要桥梁。


### 2.6.1 互相关<a id='math:sec:cross_correlation'></a>

互相关与卷积非常接近，但它问的是另一个问题：当一个信号相对于另一个信号发生平移时，两者在每个延迟量上有多相似？因此，卷积更像“系统响应如何叠加输入”，而相关更像“两个信号在不同相对位移下的匹配程度”。

对干涉测量来说，这个区别极其重要，因为相关器真正计算的不是卷积，而是两个接收电场之间的互相关。正是这个量在适当归一化和几何解释之后，对应于我们后面要使用的复可见度。


<a id='math:eq:7_001'></a><!--\label{math:eq:7_001}-->

$$
\star: \left\{f\,|\, f:\mathbb{R}\rightarrow \mathbb{C}\right\}\,\times\, \left\{f\,|\, f:\mathbb{R}\rightarrow \mathbb{C}\right\} \rightarrow \left\{f\,|\, f:\mathbb{R}\rightarrow \mathbb{C}\right\}\\
\begin{aligned}
(f\star g)(x) \,&=\, ({f_-}^*\circ g)(x)\\
&=\, \int_{-\infty}^{+\infty} f^*(t-x)\,g(t)\,dt\\
&\underset{t^\prime = t-x}{=}\, \int_{-\infty}^{+\infty} f^*(t^\prime)\,g(t^\prime+x)\,dt^\prime\\
\end{aligned}\qquad
$$

其中 $f_-(x) = f(-x)$。也就是说，互相关可以看成“先把一个函数反射并取共轭，再与另一个函数做卷积”。在多维情形下，这一结构完全保留：


 <a id='math:eq:7_002'></a><!--\label{math:eq:7_002}-->

$$
\star: \left\{f\,|\, f:\mathbb{R}^n\rightarrow \mathbb{C}\right\}\,\times\, \left\{f\,|\, f:\mathbb{R}^n\rightarrow \mathbb{C}\right\} \rightarrow \left\{f\,|\, f:\mathbb{R}^n\rightarrow \mathbb{C}\right\}\, \quad n \in \mathbb{N} \\
\begin{aligned}
(f\star g)(x_1,\ldots,x_n ) \,&=\, (f\star g)({\bf x})\\
&=\, ({f_-}^*\circ g)(x)\\
&=\, \int_{-\infty}^{+\infty} \ldots \int_{-\infty}^{+\infty} f^*(t_1-x_1, \ldots , t_n-x_n)\,g(t_1, \ldots, t_n) \,d^nt\\
\,&=\, \int_{-\infty}^{+\infty} f^*({\bf t}-{\bf x})\,g({\bf t}) \,d^nt\\
\,&=\, \int_{-\infty}^{+\infty} f^*({\bf t})\,g({\bf t}+{\bf x}) \,d^nt\\
\end{aligned}
$$

互相关还有一个值得记住的对称性质：


 <a id='math:eq:7_003'></a><!--\label{math:eq:7_003}-->

$$
\begin{aligned}
(f\star g)(x) \,&=\, \int_{-\infty}^{+\infty} f^*(t-x)\,g(t) \,dt\\
\,&=\, \int_{-\infty}^{+\infty} [f(t-x)\,g^*(t)]^* \,dt\\
\,&=\, \left(\int_{-\infty}^{+\infty} f(t-x)\,g^*(t) \,dt\right)^*\\
\,&\underset{t^\prime = t-x}{=}\, \left(\int_{-\infty}^{+\infty} f(t^\prime)\,g^*(t^\prime+x) \,dt^\prime\right)^*\\
\,&=\, \left(g\star f\right)_-^*\\
\end{aligned}
\qquad
$$

**如果 $f$ 或 $g$ 中有一个既是偶函数又是实值函数，那么互相关与卷积相同。**

这个结论在物理上很直观：若响应函数本身没有方向偏置、同时又不携带额外相位信息，那么“滑动匹配”与“滑动叠加”会给出同样的结果。


### 2.6.2 自相关<a id='math:sec:auto_correlation'></a>

自相关则是把互相关作用在同一个函数上。它回答的问题是：一个信号与其自身在不同位移下的相似程度如何？因此，自相关天然携带尺度、周期性和相关长度等信息。


<a id='math:eq:7_004'></a><!--\label{math:eq:7_004}-->

$$
R: \left\{f\,|\, f:\mathbb{R}\rightarrow \mathbb{C}\right\}\,\times\, \left\{f\,|\, f:\mathbb{R}\rightarrow \mathbb{C}\right\} \rightarrow \left\{f\,|\, f:\mathbb{R}\rightarrow \mathbb{C}\right\}\\
\begin{aligned}
R\{f\}(x) \,&=\, (f\star f)(x)\\
&=\, (f_-^*\circ f)(x)\\
&=\, \int_{-\infty}^{+\infty} f^*(t-x)\,f(t)\,dt\\
&\underset{t^\prime = t-x}{=}\, \int_{-\infty}^{+\infty} f^*(t^\prime)\,f(t^\prime+x)\,dt^\prime\\
\end{aligned}\qquad \text{.}
$$

在很多情况下，研究函数的自相关性质比直接研究函数本身更容易。后面当我们把频谱与功率谱联系起来时，自相关的重要性会进一步凸显，这正是维纳-辛钦定理在信号分析和射电数据处理中如此常见的原因。


***

* 下一节：[2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)
